___
<a href='https://honglab.ai'><p style="text-align:center;"><img src='https://lh3.googleusercontent.com/lY3ySXooSmwsq5r-mRi7uiypbo0Vez6pmNoQxMFhl9fmZJkRHu5lO2vo7se_0YOzgmDyJif9fi4_z0o3ZFdwd8NVSWG6Ea80uWaf3pOHpR4GHGDV7kaFeuHR3yAjIJjDgfXMxsvw=w2400'  class="center" width="50%" height="50%"/></p></a>
___
<center><em>Content Copyright by HongLab, Inc.</em></center>

# 새 섹션

In [10]:
# 1. Korpora 설치 및 데이터 로드
from Korpora import Korpora

kowikitext = Korpora.load('kowikitext')

# 2. train 데이터를 train.txt 파일로 저장
output_path = 'kowikitext_train.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    for text in kowikitext.train.texts:
        f.write(text + '\n')

print("저장 완료! 파일 크기 확인 중...")
!ls -lh kowikitext_train.txt




    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : Hyunjoong Kim lovit@github
    Repository : https://github.com/lovit/kowikitext
    References :

    한국어 위키피디아의 덤프 데이터를 바탕을 제작한 wikitext 형식의 텍스트 파일입니다.
    학습 및 평가를 위하여 위키페이지 별로 train (99%), dev (0.5%), test (0.5%) 로 나뉘어져있습니다.


    # License
    CC-BY-SA 3.0 which kowiki dump dataset is licensed



[kowikitext] download kowikitext_20200920.train.zip: 494MB [00:16, 30.2MB/s]                              


unzip C:\Users\wirew\Korpora\kowikitext\kowikitext_20200920.train


[kowikitext] download kowikitext_20200920.test.zip: 2.51MB [00:00, 4.24MB/s]                            


unzip C:\Users\wirew\Korpora\kowikitext\kowikitext_20200920.test


[kowikitext] download kowikitext_20200920.dev.zip: 2.37MB [00:00, 11.8MB/s]                            

unzip C:\Users\wirew\Korpora\kowikitext\kowikitext_20200920.dev


저장 완료! 파일 크기 확인 중...


'ls'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


#### 훈련 데이터 준비

준비한 텍스트 파일을 읽어 들여서 정리한 후에 앞에 cleaned_가 붙은 파일 이름으로 정리합니다.
> 예시) alice.txt &rarr; cleaned_alice.txt

- 캐글 해리포터 책 - [Harry Potter Books](https://www.kaggle.com/datasets/shubhammaindola/harry-potter-books?select=02+Harry+Potter+and+the+Chamber+of+Secrets.txt)
- 캐글 앨리스 책 - [alice.txt](https://www.kaggle.com/datasets/leelatte/alicetxt)
- 훈련 데이터나 가중치는 제가 배포하지 않습니다. 직접 다운받거나 준비하셔야합니다.

In [20]:
import re

def clean_text(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        book_text = file.read()

    cleaned_text = re.sub(r'\n+', ' ', book_text) # 줄바꿈을 빈칸으로 변경
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text) # 여러 빈칸을 하나의 빈칸으로

    print("cleaned_" + filename, len(cleaned_text), "characters") # 글자 수 출력

    with open("cleaned_" + filename, 'w', encoding='utf-8') as file:
        file.write(cleaned_text)

filenames_list = ["cleaned_kowikitext_train.txt"]

for filename in filenames_list:
    clean_text(filename)

cleaned_cleaned_kowikitext_train.txt 5052183 characters


In [19]:
from pathlib import Path
import re

NUM_PATTERN = re.compile(r"[0-9]+")
DROP_SYMBOLS_KEEP_PUNCT = re.compile(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9.,?!\s]")
DROP_EXTRA_PUNCT = re.compile(r"[\"'`“”‘’〈〉《》「」『』()\[\]{}<>/\\|@#$%^&*_+=~-]+")
DROP_SYMBOLS_STRICT = re.compile(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z\s]")
MULTI_SPACE = re.compile(r"[ \t]+")
MULTI_NEWLINE = re.compile(r"\n{3,}")


def clean_text(text: str, keep_basic_punct: bool = True) -> str:
    text = NUM_PATTERN.sub(" ", text)
    if keep_basic_punct:
        text = DROP_SYMBOLS_KEEP_PUNCT.sub(" ", text)
        text = DROP_EXTRA_PUNCT.sub(" ", text)
    else:
        text = DROP_SYMBOLS_STRICT.sub(" ", text)

    text = MULTI_SPACE.sub(" ", text)
    text = MULTI_NEWLINE.sub("\n\n", text)
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if len(line) > 0]
    return "\n".join(lines).strip()


def run_cleaning(
    input_path: Path,
    output_path: Path,
    keep_basic_punct: bool = True,
    max_lines: int | None = None,
    max_chars: int | None = None,
    chunk_lines: int = 20_000,
) -> None:
    if not input_path.exists():
        raise FileNotFoundError(f"입력 파일을 찾을 수 없습니다: {input_path.resolve()}")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    read_lines = 0
    read_chars = 0
    written_blocks = 0
    buffer: list[str] = []
    reached_limit = False

    with input_path.open("r", encoding="utf-8", errors="ignore") as fin, output_path.open(
        "w", encoding="utf-8"
    ) as fout:
        for line in fin:
            if max_lines is not None and read_lines >= max_lines:
                reached_limit = True
                break
            if max_chars is not None and read_chars >= max_chars:
                reached_limit = True
                break
            if max_chars is not None and (read_chars + len(line)) > max_chars:
                line = line[: max_chars - read_chars]
                reached_limit = True

            read_lines += 1
            read_chars += len(line)
            buffer.append(line)

            if len(buffer) >= chunk_lines or reached_limit:
                cleaned = clean_text("".join(buffer), keep_basic_punct=keep_basic_punct)
                buffer.clear()
                if cleaned:
                    fout.write(cleaned + "\n")
                    written_blocks += 1

            if reached_limit:
                break

        if buffer:
            cleaned = clean_text("".join(buffer), keep_basic_punct=keep_basic_punct)
            if cleaned:
                fout.write(cleaned + "\n")
                written_blocks += 1

    out_size_mb = output_path.stat().st_size / (1024 * 1024) if output_path.exists() else 0.0
    print("전처리 완료!")
    print(f"- 입력 파일: {input_path}")
    print(f"- 출력 파일: {output_path} ({out_size_mb:.2f} MB)")
    print(f"- 읽은 줄 수: {read_lines:,} / 읽은 글자 수: {read_chars:,} / 저장된 블록 수: {written_blocks:,}")


# 여기서 원하는 값으로 수정 후 실행
run_cleaning(
    input_path=Path(r"kowikitext_train.txt"),
    output_path=Path(r"cleaned_kowikitext_train.txt"),
    keep_basic_punct=True,
    max_lines=100_000,   # 전체는 None
    max_chars=None,
    chunk_lines=20_000,
)

전처리 완료!
- 입력 파일: kowikitext_train.txt
- 출력 파일: cleaned_kowikitext_train.txt (10.91 MB)
- 읽은 줄 수: 100,000 / 읽은 글자 수: 5,678,343 / 저장된 블록 수: 5


#### 토큰화

UTF-8 BPE(Bype Pair Encoding)

In [1]:
import tiktoken # pip install tiktoken

tokenizer = tiktoken.get_encoding("o200k_base")

text = "Harry Potter was a wizard."

tokens = tokenizer.encode(text)

print("글자수:", len(text), "토큰수", len(tokens))
print(tokens)
print(tokenizer.decode(tokens))
for t in tokens:
    print(f"{t}\t -> {tokenizer.decode([t])}")

글자수: 26 토큰수 6
[78006, 45666, 673, 261, 67537, 13]
Harry Potter was a wizard.
78006	 -> Harry
45666	 ->  Potter
673	 ->  was
261	 ->  a
67537	 ->  wizard
13	 -> .


In [ ]:
# from transformers import AutoTokenizer # pip install transformers

# tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct")  # KoGPT2 사용
# # tokenizer = AutoTokenizer.from_pretrained("skt/kogpt2-base-v2")  # KoGPT2 사용

# print("Vocab size :", len(tokenizer))

# text = "대사께서는 도(道)를 얻은 모양이구려."

# tokens = tokenizer.encode(text)

# print(len(text), len(tokens))
# print(tokens)
# print(tokenizer.decode(tokens))

In [ ]:
for char in text:
    token_ids = tokenizer.encode(char)     # 한 글자씩 인코딩(토큰화)
    decoded = tokenizer.decode(token_ids)  # 한 글자씩 디코딩
    print(f"{char} -> {token_ids} -> {decoded}")

H -> [39] -> H
a -> [64] -> a
r -> [81] -> r
r -> [81] -> r
y -> [88] -> y
  -> [220] ->  
P -> [47] -> P
o -> [78] -> o
t -> [83] -> t
t -> [83] -> t
e -> [68] -> e
r -> [81] -> r
  -> [220] ->  
w -> [86] -> w
a -> [64] -> a
s -> [82] -> s
  -> [220] ->  
a -> [64] -> a
  -> [220] ->  
w -> [86] -> w
i -> [72] -> i
z -> [89] -> z
a -> [64] -> a
r -> [81] -> r
d -> [67] -> d
. -> [13] -> .


#### 데이터로더(DataLoader)

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, txt, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # token_ids = tokenizer.encode("<|endoftext|>" + txt, allowed_special={"<|endoftext|>"})
        token_ids = tokenizer.encode(txt)

        print("# of tokens in txt:", len(token_ids))

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length] #input_chunk와 target_chunk가 1개씩 차이나는 이유는 다음 단어를 예측 하기 위해서이다.
            target_chunk = token_ids[i + 1: i + max_length + 1] #ex) hello, -> target_cunk에서는 hello, world가 나와야함.
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# with open("cleaned_한글문서.txt", 'r', encoding='utf-8-sig') as file: # 선택: -sig를 붙여서 BOM 제거
with open("cleaned_cleaned_kowikitext_train.txt", 'r', encoding='utf-8-sig') as file: # 선택: -sig를 붙여서 BOM 제거
    txt = file.read() # 파일을 불러와서 text를 txt 변수에 저장.
    print(len(txt))
    le = int(input("어느정도 길이 : "))
    txt = txt[0:le]



dataset = MyDataset(txt, max_length = 256, stride = 256) # txt 변수에 저장된 것 + 최대 길이 및 stride(폭 설정) 설정
torch.save(dataset, "hp_dataset.pt")
print("데이터셋 저장 완료!")



5052183
# of tokens in txt: 2972555
데이터셋 저장 완료!


In [23]:
import torch
from torch.utils.data import Dataset, DataLoader

# 주의: torch.load로 클래스 객체를 불러올 때는 MyDataset 클래스 정의가 반드시 필요합니다.
class MyDataset(Dataset):
    def __init__(self, txt, max_length, stride):
        # ... 기존 코드와 동일 ...
        pass
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# 파일에서 불러와 다시 dataset 변수에 저장
# (PyTorch 2.6 이상에서는 클래스 객체 로드를 위해 weights_only=False 설정 필요)
dataset = torch.load("hp_dataset.pt", weights_only=False)

# 바로 DataLoader에 연결하여 사용
train_loader = DataLoader(dataset, batch_size=8, shuffle=True, drop_last=True)

In [31]:
dataiter = iter(train_loader)

x, y = next(dataiter) # x는 입력할 것, y는 타깃.

print(next(dataiter)) #텐서들이 나옴.
print(tokenizer.decode(x[0].tolist())) #복화화 과정. 디코딩해서 사람 눈에 보이는 단어로 만든다.
print(tokenizer.decode(y[0].tolist()))


[tensor([[  5637, 184765,   2888,  ...,  70682, 169344,  66603],
        [  2186,  28647,   5947,  ...,   3956, 108833, 187917],
        [ 88440,  34950,  32518,  ...,  14377,  13611,  55432],
        ...,
        [   234, 116286, 114956,  ...,  14108,  46947,  32077],
        [ 12753,  10599,  65838,  ...,  26387,  12809,   4740],
        [ 11440,   3408,  40160,  ..., 188039,  11630,  87723]]), tensor([[184765,   2888,    244,  ..., 169344,  66603,   8562],
        [ 28647,   5947, 103921,  ..., 108833, 187917,  11890],
        [ 34950,  32518,  14844,  ...,  13611,  55432,   5537],
        ...,
        [116286, 114956,  48744,  ...,  46947,  32077,   2276],
        [ 10599,  65838,   3107,  ...,  12809,   4740,   9023],
        [  3408,  40160,  67552,  ...,  11630,  87723, 197575]])]
커플 에피소드 목록은, ChongDae님이 저작권침해를 이유로 삭제한 문서가 아닙니다. 당시 ChongaDae님이 삭제한 버전은 이것 삭제되어 현재는 일반사용자는 현재 볼 수 없는 상태 입니다. 관인생략님은 ChongaDae님이 삭제한 버전에 관하여 그 인용이 과하지 않고 법적 판례가 존재하지 않는다고 위에서 주장하셨는데, 삭제된 버전은 홈페이지 미리보기 내

위 코드는 ai에게 넣을 때, 입력 단어를 넣었을 때, 출력 단어가 나오게 학습시키는 코드를 시각적으로 보여준다.

#### 뉴럴네트워크 모델 정의

모델 정의는 교재 "[Build a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)"에서 제공하는 [예제 코드](https://github.com/rasbt/LLMs-from-scratch)를 약간 수정하였습니다.


In [32]:
# 모델을 정의할 때 사용하는 상수들

VOCAB_SIZE = tokenizer.n_vocab # 50257 Tiktoken 토큰에 사전이 몇개가 있나.
#VOCAB_SIZE = len(tokenizer) # AutoTokenizer
CONTEXT_LENGTH = 256  # Shortened context length (orig: 1024)
EMB_DIM = 768  # Embedding dimension
NUM_HEADS = 12  # Number of attention heads
NUM_LAYERS = 12  # Number of layers
DROP_RATE = 0.1  # Dropout rate
QKV_BIAS = False  # Query-key-value bias
print(CONTEXT_LENGTH)

256


In [33]:
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        assert d_out % NUM_HEADS == 0, "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.head_dim = d_out // NUM_HEADS

        self.W_query = nn.Linear(d_in, d_out, bias=QKV_BIAS)
        self.W_key = nn.Linear(d_in, d_out, bias=QKV_BIAS)
        self.W_value = nn.Linear(d_in, d_out, bias=QKV_BIAS)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(DROP_RATE)
        self.register_buffer('mask', torch.triu(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, NUM_HEADS, self.head_dim)
        values = values.view(b, num_tokens, NUM_HEADS, self.head_dim)
        queries = queries.view(b, num_tokens, NUM_HEADS, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(EMB_DIM, 4 * EMB_DIM),
            GELU(),
            nn.Linear(4 * EMB_DIM, EMB_DIM),
        )

    def forward(self, x):
        return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=EMB_DIM,
            d_out=EMB_DIM)

        self.ff = FeedForward()
        self.norm1 = LayerNorm(EMB_DIM)
        self.norm2 = LayerNorm(EMB_DIM)
        self.drop_shortcut = nn.Dropout(DROP_RATE)

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LENGTH, EMB_DIM)
        self.drop_emb = nn.Dropout(DROP_RATE)

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock() for _ in range(NUM_LAYERS)])

        self.final_norm = LayerNorm(EMB_DIM)
        self.out_head = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
import gc

# train_loader 생성 완료 후 실행
del txt
gc.collect()


NameError: name 'txt' is not defined

### '이 코드에서 사용된 파이토치 사용법'

### 1. `nn.Module`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | PyTorch 신경망 기본 클래스 |
| **역할** | 모든 신경망 모듈의 뼈대가 되는 클래스로, `super().__init__()`을 통해 내부 가중치와 서브 모듈을 올바르게 등록 및 관리 |
| **코드 예시** | `class MultiHeadAttention(nn.Module):` |

---

### 2. `nn.Parameter`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 학습 가능한 텐서 (Parameter) |
| **역할** | 일반 텐서와 달리 `nn.Module`에 속할 때 **옵티마이저(Optimizer)가 업데이트할 대상(학습 파라미터)**으로 자동 등록 |
| **코드 예시** | `self.scale = nn.Parameter(torch.ones(emb_dim))` |

---

### 3. `nn.Linear`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 선형 변환 계층 (Linear / Dense Layer) |
| **역할** | 입력 차원(`d_in`)을 출력 차원(`d_out`)으로 선형 변환($y = xW^T + b$)합니다. `bias=False`를 주면 편향을 제외 |
| **코드 예시** | `self.W_query = nn.Linear(d_in, d_out, bias=QKV_BIAS)` |

---

### 4. `nn.Embedding`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 임베딩 룩업 테이블 (Embedding Table) |
| **역할** | 정수 형태의 토큰 ID를 지정된 차원의 고밀도 벡터(Dense Vector)로 변환 |
| **코드 예시** | `self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)` |

---

### 5. `nn.Dropout`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 과적합 방지 규제(Regularization) 레이어 |
| **역할** | 학습 과정에서 무작위로 일부 뉴런의 활성화를 0으로 만들어 모델이 과적합되는 것을 방지 |
| **코드 예시** | `self.dropout = nn.Dropout(DROP_RATE)` |

---

### 6. `nn.Sequential`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 순차적 모듈 컨테이너 |
| **역할** | 여러 개의 신경망 레이어를 순서대로 실행할 수 있도록 하나로 묶는다. |
| **코드 예시** | `self.layers = nn.Sequential(nn.Linear(...), GELU(), ...)` |

---

### 7. `torch.triu`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 텐서 조작 함수 (Upper Triangular Matrix) |
| **역할** | 행렬의 주대각선 기준 상단 삼각형 부분만 남기고 나머지는 0으로 채운다. GPT 등에서 미래 토큰을 가리는 인과적 마스크 제작에 사용 |
| **코드 예시** | `torch.triu(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH), diagonal=1)` |

---

### 8. `register_buffer`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 상태 텐서 등록 함수 |
| **역할** | 학습되지는 않지만(`requires_grad=False`), 모델의 `state_dict`에 함께 저장되며 GPU/CPU 장치 이동이 자동으로 동기화되는 텐서를 등록 |
| **코드 예시** | `self.register_buffer('mask', ...)` |

---

### 9. `view()` / `reshape()`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 텐서 차원 변경 함수 |
| **역할** | 데이터의 총 개수를 유지한 채 텐서의 형태(Shape)를 변환 (멀티 헤드 어텐션에서 헤드별로 차원을 쪼갤 때 사용) |
| **코드 예시** | `keys = keys.view(b, num_tokens, NUM_HEADS, self.head_dim)` |

---

### 10. `transpose()`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 텐서 차원 교환 함수 |
| **역할** | 지정한 두 차원의 순서를 서로 맞바꾼다. 행렬 곱셈이나 어텐션 연산을 위해 차원 배치를 바꿀 때 필수 |
| **코드 예시** | `keys = keys.transpose(1, 2)` |

---

### 11. `@` 연산자
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 행렬 곱셈(Matrix Multiplication) 연산자 |
| **역할** | 파이토치 텐서 간의 행렬 곱을 수행 (`torch.matmul`과 동일한 기능) |
| **코드 예시** | `attn_scores = queries @ keys.transpose(2, 3)` |

---

### 12. `masked_fill_()`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 인플레이스 조건부 값 변경 함수 |
| **역할** | 뒤에 붙은 언더바(`_`)는 **인플레이스(In-place)** 연산을 의미하며, 새로운 객체를 만들지 않고 원본 텐서를 직접 수정 마스크 위치에 특정 값(`-torch.inf`)을 채운다. |
| **코드 예시** | `attn_scores.masked_fill_(mask_bool, -torch.inf)` |

---

### 13. `torch.softmax()`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 활성화 함수 (Softmax) |
| **역할** | 로짓(Logit) 값들을 입력받아 모든 값의 합이 1이 되는 확률 분포 형태로 변환 |
| **코드 예시** | `attn_weights = torch.softmax(attn_scores / ..., dim=-1)` |

---

### 14. `torch.arange()`
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 순차적 정수 텐서 생성 함수 |
| **역할** | 지정한 범위 내에서 일정한 간격의 정수 텐서를 생성 (위치 인덱스 생성 시 주로 사용) |
| **코드 예시** | `torch.arange(seq_len, device=in_idx.device)` |

---

### 15. `assert` (파이썬 기본 문법)
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 조건 검증 문법 (Assertion) |
| **역할** | 지정한 조건식이 참이 아닐 경우 `AssertionError`를 발생시켜 잘못된 설정이나 입력을 사전에 차단 |
| **코드 예시** | `assert d_out % NUM_HEADS == 0, "d_out must be divisible..."` |

---

### 16. `*` (언패킹 연산자, Unpacking)
| 구분 | 내용 |
| :--- | :--- |
| **분류** | 파이썬 컨테이너 언패킹 문법 |
| **역할** | 리스트나 튜플 내부의 요소들을 개별 인자로 풀어헤쳐서 전달 (동일한 트랜스포머 블록을 여러 개 생성해 `nn.Sequential`에 넣을 때 사용) |
| **코드 예시** | `*[TransformerBlock() for _ in range(NUM_LAYERS)]` |

#### 훈련

In [34]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = "cpu"
print(device)

torch.manual_seed(123)
model = GPTModel()
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

cuda


In [9]:
import torch
print(torch.__version__)


2.5.1+cu121


통합


In [4]:
import torch
import torch.nn as nn

# 1. 경량화 상수 설정
VOCAB_SIZE = 200019
CONTEXT_LENGTH = 256
EMB_DIM = 256
NUM_HEADS = 8
NUM_LAYERS = 6
DROP_RATE = 0.1
QKV_BIAS = False

# 2. 클래스 재정의 (전역 변수 꼬임 방지)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.head_dim = d_out // NUM_HEADS
        self.W_query = nn.Linear(d_in, d_out, bias=QKV_BIAS)
        self.W_key = nn.Linear(d_in, d_out, bias=QKV_BIAS)
        self.W_value = nn.Linear(d_in, d_out, bias=QKV_BIAS)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(DROP_RATE)
        self.register_buffer('mask', torch.triu(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x).view(b, num_tokens, NUM_HEADS, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, NUM_HEADS, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, NUM_HEADS, self.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = self.dropout(torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1))
        context_vec = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * ((x - mean) / torch.sqrt(var + self.eps)) + self.shift

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(EMB_DIM, 4 * EMB_DIM),
            nn.GELU(),
            nn.Linear(4 * EMB_DIM, EMB_DIM),
        )

    def forward(self, x):
        return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att = MultiHeadAttention(EMB_DIM, EMB_DIM)
        self.ff = FeedForward()
        self.norm1 = LayerNorm(EMB_DIM)
        self.norm2 = LayerNorm(EMB_DIM)
        self.drop_shortcut = nn.Dropout(DROP_RATE)

    def forward(self, x):
        x = x + self.drop_shortcut(self.att(self.norm1(x)))
        x = x + self.drop_shortcut(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LENGTH, EMB_DIM)
        self.drop_emb = nn.Dropout(DROP_RATE)
        self.trf_blocks = nn.Sequential(*[TransformerBlock() for _ in range(NUM_LAYERS)])
        self.final_norm = LayerNorm(EMB_DIM)
        self.out_head = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        x = self.drop_emb(self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device)))
        x = self.trf_blocks(x)
        return self.out_head(self.final_norm(x))

# 3. 모델 생성 및 GPU 이동 (Weight Tying 적용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPTModel()
model.out_head.weight = model.tok_emb.weight
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

print("완료! GPU 메모리 사용량:", f"{torch.cuda.memory_allocated() / 1024**2:.1f} MB")


완료! GPU 메모리 사용량: 215.8 MB


In [11]:
tokens_seen, global_step = 0, -1

losses = []

for epoch in range(5):
    model.train()  # Set model to training mode

    epoch_loss = 0
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad() # Reset loss gradients from previous batch iteration
        input_batch, target_batch = input_batch.to(device), target_batch.to(device)

        logits = model(input_batch)
        loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
        epoch_loss += loss.item()
        loss.backward() # Calculate loss gradients
        optimizer.step() # Update model weights using loss gradients
        tokens_seen += input_batch.numel()
        global_step += 1

        if global_step % 1000 == 0:
            print(f"Tokens seen: {tokens_seen}")
        # Optional evaluation step

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch: {epoch + 1}, Loss: {avg_loss}")
    torch.save(model.state_dict(), "model_" + str(epoch + 1).zfill(3) + ".pth")

# 주의: 여기서는 편의상 모든 데이터를 train에 사용하였습니다.
#      ML에서는 일부 데이터를 validation에 사용하는 것이 일반적입니다.

OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 7.96 GiB of which 0 bytes is free. Of the allocated memory 14.24 GiB is allocated by PyTorch, and 130.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

# 로딩창 포함

In [37]:
from torch.utils.data import DataLoader, Subset

# 예: 전체 데이터 중 앞부분 10% (약 1/10)만 사용
keep_ratio = 0.02  # 0.05(5%), 0.1(10%), 0.2(20%) 등으로 조절
num_samples = max(1, int(len(dataset) * keep_ratio))

sub_dataset = Subset(dataset, range(num_samples))

# 기존 batch_size(예: 8 또는 16) 그대로 사용
train_loader = DataLoader(sub_dataset, batch_size=8, shuffle=True, drop_last=True)

print("원래 샘플 수:", len(dataset))
print("줄어든 샘플 수:", len(sub_dataset))
print("1 에포크 스텝 수:", len(train_loader))

원래 샘플 수: 11611
줄어든 샘플 수: 232
1 에포크 스텝 수: 29


In [38]:
from tqdm.auto import tqdm
import torch

epochs = 2
tokens_seen, global_step = 0, -1
losses = []

for epoch in range(epochs):
    model.train()

    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=True)

    for input_batch, target_batch in progress_bar:
        optimizer.zero_grad(set_to_none=True)
        input_batch, target_batch = input_batch.to(device, non_blocking=True), target_batch.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_batch)
            loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

        epoch_loss += loss.item()
        loss.backward()
        optimizer.step()
        tokens_seen += input_batch.numel()
        global_step += 1

        if global_step % 10 == 0:
            progress_bar.set_postfix(loss=f"{loss.item():.4f}", tokens=f"{tokens_seen:,}")

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch: {epoch + 1}/{epochs}, Average Loss: {avg_loss:.4f}")

    # 각 에포크가 끝날 때마다 체크포인트 저장
    ckpt_path = f"model_epoch_{epoch + 1}.pt"
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "avg_loss": avg_loss,
        "tokens_seen": tokens_seen,
    }, ckpt_path)
    print(f"저장 완료: {ckpt_path}")

Epoch 1/2: 100%|██████████| 29/29 [05:14<00:00, 10.86s/it, loss=7.1620, tokens=43,008]


Epoch: 1/2, Average Loss: 8.4521
저장 완료: model_epoch_1.pt


Epoch 2/2: 100%|██████████| 29/29 [05:58<00:00, 12.36s/it, loss=6.8317, tokens=104,448]


Epoch: 2/2, Average Loss: 6.9393
저장 완료: model_epoch_2.pt


In [10]:
import gc
import torch
from tqdm.auto import tqdm

# 1. 이전 실행에서 남은 캐시 및 피크 메모리 통계 초기화
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# 2. 학습 설정 (8GB VRAM 환경에 맞춰 accum_steps, epochs 조절)
epochs = 5
accum_steps = 4  # batch_size를 줄인 만큼 늘려서 기존 유효 배치 크기 유지 (1이면 누적 안 함)
use_bf16 = (device == "cuda") or (
    hasattr(device, "type") and device.type == "cuda"
)

tokens_seen, global_step = 0, -1
losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=True)

    for step, (input_batch, target_batch) in enumerate(progress_bar):
        input_batch = input_batch.to(device, non_blocking=True)
        target_batch = target_batch.to(device, non_blocking=True)

        # RTX 50 시리즈(Blackwell)에서 메모리 절감에 유리한 bfloat16 적용
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(input_batch)
            loss = torch.nn.functional.cross_entropy(
                logits.flatten(0, 1),
                target_batch.flatten()
            )
            scaled_loss = loss / accum_steps

        # 원래 스케일의 손실 기록
        epoch_loss += loss.item()

        # 역전파 (누적 스텝 단위로 파라미터 업데이트)
        scaled_loss.backward()

        is_accum_boundary = ((step + 1) % accum_steps == 0) or ((step + 1) == len(train_loader))
        if is_accum_boundary:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        tokens_seen += input_batch.numel()

        if global_step >= 0 and global_step % 10 == 0:
            peak_gb = (
                torch.cuda.max_memory_allocated() / (1024 ** 3)
                if use_bf16 else 0.0
            )
            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                tokens=f"{tokens_seen:,}",
                peak_GB=f"{peak_gb:.2f}"
            )

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    peak_gb = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
        if use_bf16 else 0.0
    )
    print(f"Epoch: {epoch + 1}/{epochs} | Average Loss: {avg_loss:.4f} | Peak VRAM: {peak_gb:.2f} GiB")

Epoch 1/5:   0%|          | 0/286 [00:12<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 384.00 MiB. GPU 0 has a total capacity of 7.96 GiB of which 0 bytes is free. Of the allocated memory 13.94 GiB is allocated by PyTorch, and 249.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [39]:
import matplotlib.pyplot as plt

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.show()

# 보충: validation loss를 같이 그려서 비교하는 사례 https://www.geeksforgeeks.org/training-and-validation-loss-in-deep-learning/

ModuleNotFoundError: No module named 'matplotlib'

In [26]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
print("Supported archs:", torch.cuda.get_arch_list())


GPU: NVIDIA GeForce RTX 5050 Laptop GPU
Capability: (12, 0)
Supported archs: ['sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']


In [61]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("Arch list:", torch.cuda.get_arch_list())

x = torch.randn(4, 4, device="cuda")
print("Matmul test:", (x @ x).shape)

PyTorch: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 5050 Laptop GPU
Arch list: ['sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [1]:
import sys, torch

print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)
print("Built CUDA version:", torch.version.cuda)
print("Arch list:", torch.cuda.get_arch_list())


Python executable: c:\Users\wirew\OneDrive\Desktop\sungbin\.venv\Scripts\python.exe
PyTorch version: 2.11.0+cu128
Built CUDA version: 12.8
Arch list: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


#### 결과 확인

In [41]:
# 파일로 저장했던 네트워크의 가중치들 읽어들이기
model.load_state_dict(torch.load("model_epoch_2.pt", map_location=device, weights_only=True))
model.eval() # dropout을 사용하지 않음

RuntimeError: Error(s) in loading state_dict for GPTModel:
	Missing key(s) in state_dict: "tok_emb.weight", "pos_emb.weight", "trf_blocks.0.att.mask", "trf_blocks.0.att.W_query.weight", "trf_blocks.0.att.W_key.weight", "trf_blocks.0.att.W_value.weight", "trf_blocks.0.att.out_proj.weight", "trf_blocks.0.att.out_proj.bias", "trf_blocks.0.ff.layers.0.weight", "trf_blocks.0.ff.layers.0.bias", "trf_blocks.0.ff.layers.2.weight", "trf_blocks.0.ff.layers.2.bias", "trf_blocks.0.norm1.scale", "trf_blocks.0.norm1.shift", "trf_blocks.0.norm2.scale", "trf_blocks.0.norm2.shift", "trf_blocks.1.att.mask", "trf_blocks.1.att.W_query.weight", "trf_blocks.1.att.W_key.weight", "trf_blocks.1.att.W_value.weight", "trf_blocks.1.att.out_proj.weight", "trf_blocks.1.att.out_proj.bias", "trf_blocks.1.ff.layers.0.weight", "trf_blocks.1.ff.layers.0.bias", "trf_blocks.1.ff.layers.2.weight", "trf_blocks.1.ff.layers.2.bias", "trf_blocks.1.norm1.scale", "trf_blocks.1.norm1.shift", "trf_blocks.1.norm2.scale", "trf_block

In [ ]:
idx = tokenizer.encode("Dobby is") # 토큰 id의 list
idx = torch.tensor(idx).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(idx)

logits = logits[:, -1, :]

# 가장 확률이 높은 단어 10개 출력
top_logits, top_indices = torch.topk(logits, 10)
for p, i in zip(top_logits.squeeze(0).tolist(), top_indices.squeeze(0).tolist()):
    print(f"{p:.2f}\t {i}\t {tokenizer.decode([i])}")

# 가장 확률이 높은 단어 출력
idx_next = torch.argmax(logits, dim=-1, keepdim=True)
flat = idx_next.squeeze(0) # 배치 차원 제거 torch.Size([1])
out = tokenizer.decode(flat.tolist()) # 텐서를 리스트로 바꿔서 디코드
print(out)

8.57	 4385	  supposed
8.53	 257	  a
8.52	 1479	  free
8.05	 15967	  sir
7.64	 7725	  hoping
7.61	 994	  here
7.48	 429	 nt
7.39	 3501	  giving
7.39	 10291	  wanting
7.29	 407	  not
 supposed


In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
start_context = input("Start context: ")

# idx = tokenizer.encode(start_context, allowed_special={'<|endoftext|>'})
idx = tokenizer.encode(start_context)
idx = torch.tensor(idx).unsqueeze(0)

context_size = model.pos_emb.weight.shape[0]

for i in range(10):

    token_ids = generate(
        model=model,
        idx=idx.to(device),
        max_new_tokens=50,
        context_size= context_size,
        top_k=50,
        temperature=0.5
    )

    flat = token_ids.squeeze(0) # remove batch dimension
    out = tokenizer.decode(flat.tolist()).replace("\n", " ")

    print(i, ":", out)

0 : Dobby is safe sir !he squeaked .Dobby is Dobby sir !squealed Dobby .Dobby is safe to Dobby sir !Dobby is having a is Dobby sir !S Dobby sir !Harry Potter sir Dobby is
1 : Dobby is having a chance to rival Harry Potter sir !Dobby is the elf sir !Winky is freed Dobby whispered frantically .Winky is a houseelf isnt her masters elf sir sir !said Dobby sir !Harry Potter Dobby heard D
2 : Dobby is supposed to be freed for Harry Potter sir !Harry Potter is safe and Dobby is free of Winky !said Dobby earnestly .Winky Dobby is not knowing sir !said Dobby is too sir !said Dobby is !said
3 : Dobby is a sir and Dobby is free .Dobby is sir !Dobby is free and Dobby is free sir !Dobby is free to sir !Dobby is free and Winky sir !Winky sir !Dobby is free elf sir
4 : Dobby is a sir and Dobby is free sir and sir and Dobby is free .Dobby is safe sir sir !Dobby is free sir he is free and sir and Winky is free to Dobby sir .Winky sir !W is freed
5 : Dobby is a sir and his family sir and Dobby is free .

#### 보충

- 여기서 소개해드린 LLM은 한 단어씩 만들어 가는 **자동회귀(autoregressive)** LLM 이라고 합니다. (자가회귀로 번역하기도 합니다.)
- 최근에는 **디퓨전(Diffusion)** LLM 기술도 나오기 시작했습니다. 한번에 한 단어씩이 아니라 전체를 생성합니다. ([참고1](https://x.com/karpathy/status/1894923254864978091), [참고2](https://x.com/omarsar0/status/1891568386494300252))